In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

In [ ]:
# Update the path below to where you extracted the shapefile
shapefile_path = "110m_cultural/ne_110m_admin_0_countries.shp"
world = gpd.read_file(shapefile_path)

fig, ax = plt.subplots(figsize=(12, 8))
world.plot(ax=ax, color='lightgray', edgecolor='black')
ax.set_title('General World Map')
plt.show()

In [ ]:
# Example: Plot only South American countries

import matplotlib.pyplot as plt
import geopandas as gpd

# Load the shapefile (update the path if needed)
shapefile_path = "110m_cultural/ne_110m_admin_0_countries.shp"
world = gpd.read_file(shapefile_path)

# Filter for South America
south_america = world[world['CONTINENT'] == 'South America']

# Plot
fig, ax = plt.subplots(figsize=(8, 10))
south_america.plot(ax=ax, color='lightgreen', edgecolor='black')
ax.set_title('South America')
plt.show()

In [ ]:
import geopandas as gpd
import requests
from io import BytesIO
import plotly.express as px
import json

# Load world shapefile and filter for Colombia
shapefile_path = "110m_cultural/ne_110m_admin_0_countries.shp"
world = gpd.read_file(shapefile_path)
colombia = world[world['ADMIN'] == 'Colombia']

# Download protected areas polygons from ArcGIS REST service
url = "https://mapas.parquesnacionales.gov.co/arcgis/rest/services/pnn/Limites_oficiales_Poligono/MapServer/0/query?where=1%3D1&outFields=*&f=geojson"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
response.raise_for_status()
polygons = gpd.read_file(BytesIO(response.content))

# Ensure both GeoDataFrames use the same CRS
polygons = polygons.to_crs(colombia.crs)

# Convert polygons to GeoJSON dict
polygons_geojson = json.loads(polygons.to_json())

# Create interactive map with Plotly
fig = px.choropleth_mapbox(
    polygons,
    geojson=polygons_geojson,
    locations=polygons.index,
    color_discrete_sequence=["green"],
    hover_data=polygons.columns,  # Show all metadata on hover
    mapbox_style="carto-positron",
    center={"lat": 4.5, "lon": -74},
    zoom=4.5,
    opacity=0.5,
    title="Colombia Protected Areas (Interactive)"
)

# Add Colombia country border as a layer
colombia_json = colombia.__geo_interface__
fig.update_layout(
    mapbox_layers=[
        {
            "source": colombia_json,
            "type": "line",
            "color": "black",
            "line": {"width": 2},
        }
    ] + list(fig.layout.mapbox.layers)
)

fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import requests
from io import BytesIO

# Load world shapefile and filter for Colombia
shapefile_path = "110m_cultural/ne_110m_admin_0_countries.shp"
world = gpd.read_file(shapefile_path)
colombia = world[world['ADMIN'] == 'Colombia']

# Download protected areas polygons from ArcGIS REST service
url = "https://mapas.parquesnacionales.gov.co/arcgis/rest/services/pnn/Limites_oficiales_Poligono/MapServer/0/query?where=1%3D1&outFields=*&f=geojson"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
response.raise_for_status()
polygons = gpd.read_file(BytesIO(response.content))

# Plot Colombia and overlay polygons
fig, ax = plt.subplots(figsize=(10, 12))
colombia.plot(ax=ax, color='white', edgecolor='black', linewidth=1)
polygons.plot(ax=ax, color='green', alpha=0.5, edgecolor='darkgreen')
ax.set_title('Colombia with Protected Areas')
plt.axis('off')
plt.show()